Create fact table

In [0]:
df_silver_fact = spark.sql('''select * from parquet.`abfss://silver-adf@rgstorage.dfs.core.windows.net/car_sales/`''')
display(df_silver_fact)

Reading all the Dim tables

In [0]:
df_dealer = spark.sql('''select * from sales_catalogue.gold.dim_Dealer''')

df_branch = spark.sql('''select * from sales_catalogue.gold.dim_branch''')

df_model = spark.sql('''select * from sales_catalogue.gold.dim_Model''')

df_date = spark.sql('''select * from sales_catalogue.gold.dim_date''')

Bringing all the Dim keys to fact table

In [0]:
df_fact = df_silver_fact.join(df_branch, df_silver_fact.Branch_ID == df_branch.Branch_ID, "left")\
                        .join(df_dealer, df_silver_fact.Dealer_ID == df_dealer.Dealer_ID, "left")\
                        .join(df_model, df_silver_fact.Model_ID == df_model.Model_ID, "left")\
                        .join(df_date, df_silver_fact.Date_ID == df_date.Date_ID, "left")\
                        .select(df_silver_fact.Revenue, df_silver_fact.Units_Sold, df_silver_fact.Revenue_per_unit, df_branch.Dim_Branch_key, df_dealer.Dim_Dealer_key, df_model.Dim_model_key, df_date.Dim_Date_key)


In [0]:
display(df_fact)

In [0]:
df_fact.write.format('delta')\
        .mode('overwrite')\
            .option("path", "abfss://gold-adf@rgstorage.dfs.core.windows.net/car_fact_sales")\
            .saveAsTable('sales_catalogue.gold.fact_sales')